# Analiza strategija uzorkovanja za GNN trening

Merenja dopune zadatka na nacionalnoj platformi za veštačku inteligenciju, na čvoru sa karticom NVIDIA A100.

Notebook sam prolazi kroz sva tri seeda i ne traži nikakvu intervenciju. Svaki skup podataka se učitava jednom, a ne po jednom za svaki seed. Rezultat svake konfiguracije se upisuje odmah, pa se pri ponovnom pokretanju preskače ono što je već izmereno.

Metodološke odluke:

1. Graf ogbn-arxiv se pretvara u neusmeren, kako radi i zvanični OGB primer.
2. Trening i ocena koriste isti graf.
3. Graf ostaje u radnoj memoriji, a uzorkovanje se paralelizuje kroz radne procese, što je izmereno kao najbrža kombinacija na ovoj mašini.
4. Na kartici tokom treninga stoje samo model i tekući batch, pa izmerena memorija odražava mini-batch.
5. Vrh memorije se čita pre ocene, pa ne uključuje prolaz kroz pun graf.
6. ClusterGCN se trenira na fiksnom broju gradijentnih koraka, jer broj koraka po epohi zavisi od broja klastera.
7. Seed se postavlja na početku svake konfiguracije, pa su poređenja uparena.
8. Statistika stepena koristi ulazni stepen, jer se uzorkovanje širi po ulaznim potezima.

Priprema, jednom po instanci:

```
pip install torch_geometric ogb
pip install pyg_lib torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
```

In [ ]:
SEEDOVI = [42, 123, 7]


In [ ]:
import os, sys, time, json, gc, csv, random, shutil, logging
from pathlib import Path
from datetime import datetime

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torch_geometric.nn import SAGEConv
from torch_geometric.loader import NeighborLoader, ClusterData, ClusterLoader
from torch_geometric.utils import to_undirected, degree
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator
import torch_geometric.data

torch.serialization.add_safe_globals([
    torch_geometric.data.data.DataEdgeAttr,
    torch_geometric.data.data.DataTensorAttr,
    torch_geometric.data.storage.GlobalStorage,
])

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda')

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
                    handlers=[logging.StreamHandler(sys.stdout)], force=True)
log = logging.getLogger(__name__)

cc = torch.cuda.get_device_capability(0)
log.info(f'kartica {torch.cuda.get_device_name(0)}, compute capability {cc[0]}.{cc[1]}, '
         f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
log.info(f'jezgara {os.cpu_count()}, deljena memorija {shutil.disk_usage("/dev/shm").total/1e9:.0f} GB')

def postavi_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

def direktorijum(seed):
    p = Path(f'rezultati/run_{seed}')
    p.mkdir(parents=True, exist_ok=True)
    return p

def upisi_red(seed, fajl, red):
    p = direktorijum(seed) / fajl
    novi = not p.exists()
    with open(p, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(red.keys()))
        if novi:
            w.writeheader()
        w.writerow(red)

def vec_uradjeno(seed, fajl, kolona):
    p = direktorijum(seed) / fajl
    if not p.exists():
        return set()
    with open(p) as f:
        return {r[kolona] for r in csv.DictReader(f)}

## Zajednička podešavanja i model

In [ ]:
SKRIVENI, LR = 256, 0.01
EPOHA, BATCH, OCENA_SVAKIH = 50, 512, 5
UCITAVANJE = dict(num_workers=8, persistent_workers=True, prefetch_factor=4, pin_memory=True)

K2_KORAKA, K2_SLOJEVA, K2_BATCH_KLASTERA, K2_OCENA_SVAKIH = 5000, 3, 20, 250
KLASTERI = [100, 500, 1000, 2000]

K4_S, K4_UZORAKA = [15, 10, 5], 20
K4_BATCH = [32, 128, 512, 1024, 2048, 4096]

KORAK1 = [
    ('L2_mali',       2, [5, 5]),
    ('L2_standardni', 2, [15, 10]),
    ('L2_veliki',     2, [25, 20]),
    ('L3_mali',       3, [5, 5, 5]),
    ('L3_standardni', 3, [15, 10, 5]),
    ('L3_veliki',     3, [25, 15, 10]),
    ('L4_mali',       4, [5, 5, 5, 5]),
    ('L4_standardni', 4, [15, 10, 5, 5]),
    ('L4_veliki',     4, [25, 15, 10, 5]),
]


class GraphSAGE(torch.nn.Module):
    def __init__(self, ulaz, skriveni, izlaz, slojeva, dropout=0.5):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.slojeva, self.dropout = slojeva, dropout
        self.convs.append(SAGEConv(ulaz, skriveni))
        self.bns.append(torch.nn.BatchNorm1d(skriveni))
        for _ in range(slojeva - 2):
            self.convs.append(SAGEConv(skriveni, skriveni))
            self.bns.append(torch.nn.BatchNorm1d(skriveni))
        self.convs.append(SAGEConv(skriveni, izlaz))

    def forward(self, x, edge_index):
        for i in range(self.slojeva - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.convs[-1](x, edge_index)


def trening_korak(model, batch, opt, ceo_podgraf=False):
    batch = batch.to(device, non_blocking=True)
    opt.zero_grad()
    out = model(batch.x, batch.edge_index)
    if ceo_podgraf:
        maska = batch.train_mask
        if maska.sum() == 0:
            return
        gubitak = F.cross_entropy(out[maska], batch.y[maska])
    else:
        maska = batch.train_mask[:batch.batch_size]
        if maska.sum() == 0:
            return
        gubitak = F.cross_entropy(out[:batch.batch_size][maska],
                                  batch.y[:batch.batch_size][maska])
    gubitak.backward()
    opt.step()

## Učitavanje grafa ogbn-arxiv

Učitava se jednom za sva tri seeda. Citatna mreža je usmerena pa se pretvara u neusmerenu.

In [ ]:
dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='podaci')
data = dataset[0]
data.y = data.y.squeeze()
split_idx = dataset.get_idx_split()
evaluator = Evaluator(name='ogbn-arxiv')

potega_usmereno = data.num_edges
data.edge_index = to_undirected(data.edge_index)
data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[split_idx['train']] = True

ulazni = degree(data.edge_index[1], num_nodes=data.num_nodes)
log.info(f'ogbn-arxiv: {data.num_nodes:,} cvorova, {data.num_edges:,} potega '
         f'(usmereno je bilo {potega_usmereno:,})')
log.info(f'ulazni stepen: prosek {ulazni.mean():.2f}, medijana {ulazni.median():.0f}, '
         f'najveci {ulazni.max():.0f}')


@torch.no_grad()
def oceni(model):
    model.eval()
    x = data.x.to(device, non_blocking=True)
    ei = data.edge_index.to(device, non_blocking=True)
    out = model(x, ei).argmax(dim=-1, keepdim=True).cpu()
    del x, ei
    torch.cuda.empty_cache()
    return {s: evaluator.eval({'y_true': data.y[split_idx[s]].unsqueeze(1),
                               'y_pred': out[split_idx[s]]})['acc']
            for s in ['train', 'valid', 'test']}

## Korak 1: geometrija node-wise uzorkovanja

In [ ]:
for SEED in SEEDOVI:
    uradjeno = vec_uradjeno(SEED, 'korak1_graphsage.csv', 'oznaka')
    for oznaka, slojeva, S in KORAK1:
        if oznaka in uradjeno:
            continue
        postavi_seed(SEED)
        log.info('=' * 70)
        log.info(f'seed {SEED} | Korak 1 | {oznaka}: L={slojeva}, S={S}, B={BATCH}')

        loader = NeighborLoader(data, num_neighbors=S, input_nodes=split_idx['train'],
                                batch_size=BATCH, shuffle=True, **UCITAVANJE)
        model = GraphSAGE(data.num_features, SKRIVENI, dataset.num_classes, slojeva).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=LR)

        naj_val, naj_test, vrh, vremena = 0.0, float('nan'), 0.0, []
        for ep in range(EPOHA):
            model.train()
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
            t0 = time.time()
            for batch in loader:
                trening_korak(model, batch, opt)
            torch.cuda.synchronize()
            vremena.append(time.time() - t0)
            vrh = max(vrh, torch.cuda.max_memory_allocated() / 1e6)
            if (ep + 1) % OCENA_SVAKIH == 0 or ep == 0:
                a = oceni(model)
                if a['valid'] > naj_val:
                    naj_val, naj_test = a['valid'], a['test']
                log.info(f'  epoha {ep+1:3d}: val={a["valid"]*100:5.2f}% '
                         f'test={a["test"]*100:5.2f}% {vremena[-1]:5.2f}s {vrh:6.0f}MB')

        upisi_red(SEED, 'korak1_graphsage.csv', {
            'familija': 'GraphSAGE', 'oznaka': oznaka, 'slojeva': slojeva, 'S': str(S),
            'batch': BATCH, 'koraka_po_epohi': len(loader), 'ukupno_koraka': len(loader) * EPOHA,
            'naj_val': naj_val, 'naj_test': naj_test,
            'vreme_po_epohi': float(np.mean(vremena)),
            'vreme_po_koraku_ms': float(np.mean(vremena)) / len(loader) * 1000,
            'memorija_mb': vrh})
        log.info(f'seed {SEED} | {oznaka}: test={naj_test*100:.2f}%, '
                 f'{np.mean(vremena):.2f}s/epohi, {vrh:.0f}MB')

        del model, opt, loader
        gc.collect()
        torch.cuda.empty_cache()

log.info('Korak 1 zavrsen za sve seedove')

## Korak 2: subgraph-wise uzorkovanje

Trening ide na fiksnom broju gradijentnih koraka, istom za sve vrednosti K, jer broj koraka po epohi zavisi od broja klastera.

In [ ]:
for SEED in SEEDOVI:
    uradjeno = vec_uradjeno(SEED, 'korak2_clustergcn.csv', 'oznaka')
    for K in KLASTERI:
        if f'K{K}' in uradjeno:
            continue
        postavi_seed(SEED)
        log.info('=' * 70)
        log.info(f'seed {SEED} | Korak 2 | K={K}, {K2_KORAKA} koraka')

        t0 = time.time()
        cluster_data = ClusterData(data, num_parts=K, recursive=False)
        metis_vreme = time.time() - t0
        log.info(f'  METIS: {metis_vreme:.1f}s')

        loader = ClusterLoader(cluster_data, batch_size=K2_BATCH_KLASTERA, shuffle=True,
                               num_workers=UCITAVANJE['num_workers'], persistent_workers=True)
        model = GraphSAGE(data.num_features, SKRIVENI, dataset.num_classes, K2_SLOJEVA).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=LR)

        naj_val, naj_test, vrh = 0.0, float('nan'), 0.0
        trening_vreme, korak, epoha = 0.0, 0, 0
        torch.cuda.reset_peak_memory_stats()
        while korak < K2_KORAKA:
            epoha += 1
            model.train()
            for batch in loader:
                torch.cuda.synchronize()
                t0 = time.time()
                trening_korak(model, batch, opt, ceo_podgraf=True)
                torch.cuda.synchronize()
                vrh = max(vrh, torch.cuda.max_memory_allocated() / 1e6)
                trening_vreme += time.time() - t0
                korak += 1
                if korak % K2_OCENA_SVAKIH == 0:
                    a = oceni(model)
                    if a['valid'] > naj_val:
                        naj_val, naj_test = a['valid'], a['test']
                    model.train()
                    torch.cuda.reset_peak_memory_stats()
                if korak >= K2_KORAKA:
                    break

        upisi_red(SEED, 'korak2_clustergcn.csv', {
            'familija': 'ClusterGCN', 'oznaka': f'K{K}', 'slojeva': K2_SLOJEVA, 'klastera': K,
            'batch_klastera': K2_BATCH_KLASTERA, 'koraka_po_epohi': len(loader),
            'ukupno_koraka': korak, 'epoha': epoha, 'metis_s': metis_vreme,
            'naj_val': naj_val, 'naj_test': naj_test,
            'vreme_po_koraku_ms': trening_vreme / korak * 1000, 'memorija_mb': vrh})
        log.info(f'seed {SEED} | K={K}: test={naj_test*100:.2f}%, METIS={metis_vreme:.1f}s, '
                 f'{trening_vreme/korak*1000:.1f} ms/korak, {vrh:.0f}MB')

        del model, opt, loader, cluster_data
        gc.collect()
        torch.cuda.empty_cache()

log.info('Korak 2 zavrsen za sve seedove')

## Korak 4: faktor preklapanja na grafu ogbn-products

Graf arxiv se oslobađa, a products se učitava jednom za sva tri seeda.

In [ ]:
if 'data' in globals():
    del data
gc.collect()
torch.cuda.empty_cache()

dataset_p = PygNodePropPredDataset(name='ogbn-products', root='podaci')
data_p = dataset_p[0]
data_p.y = data_p.y.squeeze()
split_p = dataset_p.get_idx_split()
data_p.train_mask = torch.zeros(data_p.num_nodes, dtype=torch.bool)
data_p.train_mask[split_p['train']] = True
log.info(f'ogbn-products: {data_p.num_nodes:,} cvorova, {data_p.num_edges:,} potega')

deg_p = degree(data_p.edge_index[1], num_nodes=data_p.num_nodes).float()
zbir = deg_p.sum()
naivna, p = 1.0, 1.0
for s in K4_S:
    p *= s
    naivna += p
ef = [torch.clamp(deg_p, max=s).mean().item() if i == 0
      else (deg_p * torch.clamp(deg_p, max=s)).sum().item() / zbir.item()
      for i, s in enumerate(K4_S)]
sa_stepenom, p = 1.0, 1.0
for e in ef:
    p *= e
    sa_stepenom += p
log.info(f'granica po ciljanom cvoru: naivna {naivna:.0f}, sa stvarnim stepenima {sa_stepenom:.0f} '
         f'(efektivni fanout {[round(x,2) for x in ef]})')

In [ ]:
for SEED in SEEDOVI:
    uradjeno = vec_uradjeno(SEED, 'korak4_preklapanje.csv', 'batch')
    for B in K4_BATCH:
        if str(B) in uradjeno:
            continue
        postavi_seed(SEED)
        loader = NeighborLoader(data_p, num_neighbors=K4_S, input_nodes=split_p['train'],
                                batch_size=B, shuffle=True, **UCITAVANJE)
        model = GraphSAGE(data_p.num_features, SKRIVENI, dataset_p.num_classes, 3).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        model.train()
        torch.cuda.reset_peak_memory_stats()

        cvorova, potega, oom = [], [], False
        for i, batch in enumerate(loader):
            if i >= K4_UZORAKA:
                break
            cvorova.append(batch.num_nodes)
            potega.append(batch.num_edges)
            try:
                trening_korak(model, batch, opt)
            except torch.cuda.OutOfMemoryError:
                oom = True
                log.warning(f'seed {SEED} | B={B}: nedostatak memorije na kartici')
                torch.cuda.empty_cache()
                break

        vrh = torch.cuda.max_memory_allocated() / 1e6
        n_akt = float(np.mean(cvorova))
        upisi_red(SEED, 'korak4_preklapanje.csv', {
            'batch': B, 'n_max_naivno': B * naivna, 'n_max_stepen': B * sa_stepenom,
            'n_actual': '' if oom else n_akt,
            'eta_naivno_pct': '' if oom else (1 - n_akt / (B * naivna)) * 100,
            'eta_stepen_pct': '' if oom else (1 - n_akt / (B * sa_stepenom)) * 100,
            'potega': '' if oom else float(np.mean(potega)),
            'memorija_mb': '' if oom else vrh, 'oom': oom, 'uzoraka': len(cvorova)})
        if not oom:
            log.info(f'seed {SEED} | B={B:5d}: N={n_akt:9.0f} '
                     f'eta_naiv={(1-n_akt/(B*naivna))*100:5.1f}% '
                     f'eta_step={(1-n_akt/(B*sa_stepenom))*100:5.1f}% memorija={vrh:7.0f}MB')

        del model, opt, loader
        gc.collect()
        torch.cuda.empty_cache()

log.info('Korak 4 zavrsen za sve seedove')

## Grafikoni i sažetak po seedu

In [ ]:
for SEED in SEEDOVI:
    d = direktorijum(SEED)
    k1 = pd.read_csv(d / 'korak1_graphsage.csv')
    k2 = pd.read_csv(d / 'korak2_clustergcn.csv')
    k4 = pd.read_csv(d / 'korak4_preklapanje.csv')

    sve = pd.concat([k1.assign(config=k1['oznaka']), k2.assign(config=k2['oznaka'])],
                    ignore_index=True)
    boje = {'GraphSAGE': 'tab:blue', 'ClusterGCN': 'tab:green'}
    znak = {'GraphSAGE': 'o', 'ClusterGCN': 's'}
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    for ax, kol, xlab, nas in [
        (axes[0], 'vreme_po_koraku_ms', 'Vreme po gradijentnom koraku (ms)',
         f'Tacnost i vreme, seed {SEED}'),
        (axes[1], 'memorija_mb', 'Memorija (MB)', f'Tacnost i memorija, seed {SEED}'),
    ]:
        for fam, grp in sve.groupby('familija'):
            ax.scatter(grp[kol], grp['naj_test'] * 100, c=boje[fam], marker=znak[fam],
                       s=120, alpha=0.75, label=fam)
            for _, r in grp.iterrows():
                ax.annotate(r['config'], (r[kol], r['naj_test'] * 100),
                            textcoords='offset points', xytext=(7, 4), fontsize=8)
        ax.set_xlabel(xlab); ax.set_ylabel('Tacnost na test skupu (%)')
        ax.set_title(nas); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(d / 'korak3_tradeoff.png', dpi=150, bbox_inches='tight')
    plt.show()

    ok = k4[~k4['oom'].astype(str).str.lower().eq('true')].copy()
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.plot(ok['batch'], ok['eta_naivno_pct'], 'o-', color='tab:blue', linewidth=2,
             markersize=8, label='Preklapanje prema naivnoj granici')
    ax1.plot(ok['batch'], ok['eta_stepen_pct'], 'o--', color='tab:cyan', linewidth=2,
             markersize=8, label='Preklapanje prema granici sa stvarnim stepenima')
    ax1.set_xlabel('Velicina batch-a'); ax1.set_ylabel('Faktor preklapanja (%)')
    ax1.set_xscale('log', base=2); ax1.grid(alpha=0.3)
    ax2 = ax1.twinx()
    ax2.plot(ok['batch'], ok['memorija_mb'], 's--', color='tab:red', linewidth=2,
             markersize=8, label='Memorija (MB)')
    ax2.set_ylabel('Memorija (MB)', color='tab:red')
    ax2.tick_params(axis='y', labelcolor='tab:red')
    l1, n1 = ax1.get_legend_handles_labels()
    l2, n2 = ax2.get_legend_handles_labels()
    ax1.legend(l1 + l2, n1 + n2, loc='center right')
    plt.title(f'Preklapanje i memorija, ogbn-products, seed {SEED}')
    plt.tight_layout()
    plt.savefig(d / 'korak4_preklapanje.png', dpi=150, bbox_inches='tight')
    plt.show()

    json.dump({
        'seed': SEED, 'datum': datetime.now().isoformat(),
        'platforma': 'nacionalna platforma za vestacku inteligenciju',
        'kartica': torch.cuda.get_device_name(0),
        'compute_capability': f'{cc[0]}.{cc[1]}',
        'jezgara': os.cpu_count(),
        'deljena_memorija_gb': round(shutil.disk_usage('/dev/shm').total / 1e9, 1),
        'ucitavanje': UCITAVANJE, 'graf_neusmeren': True,
        'epoha_korak1': EPOHA, 'batch_korak1': BATCH,
        'koraka_korak2': K2_KORAKA, 'batch_klastera': K2_BATCH_KLASTERA,
        'skriveni': SKRIVENI, 'lr': LR,
    }, open(d / 'sazetak.json', 'w'), indent=2, default=str)

    print(f'--- seed {SEED} ---')
    print(k1[['oznaka', 'naj_test', 'vreme_po_epohi', 'memorija_mb']].to_string(index=False))
    print()

log.info('sve gotovo')